In [1]:
# New version of the notebook. We will rewrite all the RL code ourself.

# Imports

In [2]:
import json
import re
import torch
from datasets import load_dataset
from vllm import LLM, SamplingParams
from sympy import sympify, simplify 
from typing import Callable, List, Dict
from pathlib import Path
from cs336_alignment.drgrpo_grader import r1_zero_reward_fn

In [3]:
# Device setup (use GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# Setup

In [4]:
# First we give the prompt that we will use.

In [5]:
R1_ZERO_PROMPT = """
A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.
User: {question}
Assistant: <think>
"""

## Loading the data

In [6]:
# Cell 2: Load the Dataset
# Load GSM8K test split (1339 examples)
dataset = load_dataset("gsm8k", "main", split="test")

Using the latest cached version of the dataset since gsm8k couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'main' at /home/ubuntu/.cache/huggingface/datasets/gsm8k/main/0.0.0/cc7b047b6e5bb11b4f1af84efc572db110a51b3c (last modified on Mon Dec 29 18:08:23 2025).


In [7]:
# Let's look at a few examples from this data.

In [8]:
index0 = 12

ex1 = dataset[index0]
print(type(ex1))
print(ex1.keys())

<class 'dict'>
dict_keys(['question', 'answer'])


In [9]:
ex1_q = ex1['question']
ex1_a = ex1['answer']

In [10]:
print(ex1_q)

Carlos is planting a lemon tree. The tree will cost $90 to plant. Each year it will grow 7 lemons, which he can sell for $1.5 each. It costs $3 a year to water and feed the tree. How many years will it take before he starts earning money on the lemon tree?


In [11]:
print(ex1_a)

He makes $10.5 selling lemons each year because 7 x 1.5 = <<7*1.5=10.5>>10.5
He earns $7.5 each year from the lemon tree because 10.5 - 3 = <<10.5-3=7.5>>7.5
It will take 12 years to earn enough to pay off the tree because 90 / 7.5 = <<90/7.5=12>>12
He will make money in year 13 because 12 + 1 = <<12+1=13>>13
#### 13


In [12]:

# Prepare list of prompts and ground truths
prompts = []
ground_truths = []
for example in dataset:
    question = example["question"].strip()
    prompt = R1_ZERO_PROMPT.format(question=question)
    prompts.append(prompt)
    gt = example["answer"].split("####")[-1].strip()
    ground_truths.append(gt)

print(f"Loaded {len(prompts)} examples from GSM8K test set.")

Loaded 1319 examples from GSM8K test set.


## Loading the model

In [14]:
# Let's now load the Qwen model we will use.

In [15]:
# Load Qwen2.5-Math-1.5B with vLLM
model_path = "Qwen/Qwen2.5-Math-1.5B"
vllm_model = LLM(
    model=model_path,
    dtype="float16", 
    gpu_memory_utilization=0.7,  # Adjust if OOM errors
    tensor_parallel_size=1,  # Single GPU
)

sampling_params = SamplingParams(
    temperature=1.0,
    top_p=1.0,
    max_tokens=1024,
    stop=["</answer>"],
    include_stop_str_in_output=True 
)

print("Model loaded.")

`torch_dtype` is deprecated! Use `dtype` instead!


WARNING 01-03 15:44:14 config.py:1656] Casting torch.bfloat16 to torch.float16.
INFO 01-03 15:44:14 llm_engine.py:226] Initializing an LLM engine (v0.6.1.dev238+ge2c6e0a82) with config: model='Qwen/Qwen2.5-Math-1.5B', speculative_config=None, tokenizer='Qwen/Qwen2.5-Math-1.5B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=0, served_model_name=Qwen/Qwen2.5-Math-1.5B, use_v2_

INFO 01-03 15:44:15 selector.py:217] Cannot use FlashAttention-2 backend for Volta and Turing GPUs.
INFO 01-03 15:44:15 selector.py:116] Using XFormers backend.


/home/ubuntu/assignment5-alignment/.venv/lib/python3.12/site-packages/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
/home/ubuntu/assignment5-alignment/.venv/lib/python3.12/site-packages/xformers/ops/fmha/flash.py:344: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_bwd")


INFO 01-03 15:44:16 model_runner.py:1014] Starting to load model Qwen/Qwen2.5-Math-1.5B...
INFO 01-03 15:44:16 selector.py:217] Cannot use FlashAttention-2 backend for Volta and Turing GPUs.
INFO 01-03 15:44:16 selector.py:116] Using XFormers backend.
INFO 01-03 15:44:16 weight_utils.py:242] Using model weights format ['*.safetensors']
INFO 01-03 15:44:16 weight_utils.py:287] No model.safetensors.index.json found in remote.


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 01-03 15:44:39 model_runner.py:1025] Loading model weights took 2.8797 GB
INFO 01-03 15:44:40 gpu_executor.py:122] # GPU blocks: 11677, # CPU blocks: 9362
INFO 01-03 15:44:45 model_runner.py:1329] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 01-03 15:44:45 model_runner.py:1333] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 01-03 15:45:03 model_runner.py:1456] Graph capturing finished in 18 secs.
Model loaded.


# Zero-shot benchmark

In [15]:
# Test by hand.

In [16]:
outputs1 = vllm_model.generate(prompts, sampling_params)

Processed prompts: 100%|██████████| 1319/1319 [02:41<00:00,  8.15it/s, est. speed input: 1264.16 toks/s, output: 2250.99 toks/s]


In [17]:
print(type(outputs1))

<class 'list'>


In [19]:
print(type(outputs1[0]))

<class 'vllm.outputs.RequestOutput'>


In [29]:
outputs1[0].outputs[0].text.strip()

'cost of 16 ducks per day = 16*$1\ncost of 3 eggs for breakfast = 3*$1\ncost of 4 muffins = 4*$0\nmoney earned in a day from selling 3 eggs per day= sold price*quantity = 3*2\nmoney earned in a day from selling 4 muffins per day= sold price*quantity = 4*0\nmoney earned in a day from selling 16 - (3+4) ducks = (16 - (3+4))*2 = 11*2\nmoney earned in a day from selling 16-3-4 ducks net (after expenses) = past total money - (3*2+4*0)\nTotal money = 16*1 + 11*2 + (3*2+4*0) = 49\nAnkit [121]\n<img>/cimages/multimages/16/capture210463508685166922503.jpg</img>'

In [30]:
# Let's now test the reward function, how does it work?

In [33]:
index1 = 0

in1 = outputs1[index1].prompt
out1 = outputs1[index1].outputs[0].text.strip()
gt1 = ground_truths[index1]

In [38]:
in1

"\nA conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?\nAssistant: <think>\n"

In [39]:
out1

'cost of 16 ducks per day = 16*$1\ncost of 3 eggs for breakfast = 3*$1\ncost of 4 muffins = 4*$0\nmoney earned in a day from selling 3 eggs per day= sold price*quantity = 3*2\nmoney earned in a day from selling 4 muffins per day= sold price*quantity = 4*0\nmoney earned in a day from selling 16 - (3+4) ducks = (16 - (3+4))*2 = 11*2\nmoney earned in a day from selling 16-3-4 ducks net (after expenses) = past total money - (3*2+4*0)\nTotal money = 16*1 + 11*2 + (3*2+4*0) = 49\nAnkit [121]\n<img>/cimages/multimages/16/capture210463508685166922503.jpg</img>'

In [40]:
# This seems very demanding..

In [36]:
# On the above we need to test the answer parsing function.

reward1 = r1_zero_reward_fn(out1, gt1)

In [37]:
reward1

{'format_reward': 0.0, 'answer_reward': 0.0, 'reward': 0.0}

In [ ]:
# Code from Grok below.

In [ ]:
# Output directory to save results
OUTPUT_DIR = Path("zero_shot_results")
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
def evaluate_vllm(
    vllm_model: LLM,
    reward_fn: Callable[[str, str], Dict[str, float]],
    prompts: List[str],
    ground_truths: List[str],
    eval_sampling_params: SamplingParams,
    output_path: str = "results.json"
) -> List[Dict]:
    """
    Evaluate model on prompts, compute metrics, serialize to disk.
    Returns list of dicts with example, generation, scores.
    """
    outputs = vllm_model.generate(prompts, eval_sampling_params)
    
    results = []
    for idx, output in enumerate(outputs):
        generated_text = output.outputs[0].text.strip() 
        full_response = output.prompt + generated_text
        
        rewards = reward_fn(generated_text, ground_truths[idx])  
        
        result = {
            "prompt": prompts[idx],
            "generation": generated_text,
            "full_response": full_response,
            "ground_truth": ground_truths[idx],
            "format_reward": rewards["format_reward"],
            "answer_reward": rewards["answer_reward"]
        }
        results.append(result)
    
    # Serialize to disk
    with open(OUTPUT_DIR / output_path, "w") as f:
        json.dump(results, f, indent=4)
    
    print(f"Results saved to {OUTPUT_DIR / output_path}")
    return results

In [ ]:
# Run evaluation on full dataset (or slice for testing: prompts[:100])
results = evaluate_vllm(
    vllm_model=vllm_model,
    reward_fn=r1_zero_reward_fn,
    prompts=prompts,
    ground_truths=ground_truths,
    eval_sampling_params=sampling_params,
    output_path="gsm8k_zero_shot.json"
)

Processed prompts: 100%|██████████| 1319/1319 [02:46<00:00,  7.91it/s, est. speed input: 1226.66 toks/s, output: 2184.22 toks/s]


Results saved to zero_shot_results/gsm8k_zero_shot.json


In [21]:
# Categorize generations
category_counts = {
    "correct_both": 0,  # format=1, answer=1
    "format_ok_answer_wrong": 0,  # format=1, answer=0
    "both_wrong": 0  # format=0, answer=0
}

examples = {
    "correct_both": [],
    "format_ok_answer_wrong": [],
    "both_wrong": []
}

for res in results:
    fr = res["format_reward"]
    ar = res["answer_reward"]
    if fr == 1 and ar == 1:
        category_counts["correct_both"] += 1
        if len(examples["correct_both"]) < 10:
            examples["correct_both"].append(res)
    elif fr == 1 and ar == 0:
        category_counts["format_ok_answer_wrong"] += 1
        if len(examples["format_ok_answer_wrong"]) < 10:
            examples["format_ok_answer_wrong"].append(res)
    else:
        category_counts["both_wrong"] += 1
        if len(examples["both_wrong"]) < 10:
            examples["both_wrong"].append(res)

print("Category Counts:")
print(category_counts)

# Overall metrics
total = len(results)
accuracy = (category_counts["correct_both"] / total) * 100 if total > 0 else 0
format_rate = ((category_counts["correct_both"] + category_counts["format_ok_answer_wrong"]) / total) * 100
print(f"\nOverall Accuracy (correct both): {accuracy:.2f}%")
print(f"Format Success Rate: {format_rate:.2f}%")

# For writeup: Inspect examples (observe at least 10 per category if available)
print("\nExamples where both correct:")
for ex in examples["correct_both"]:
    print(f"Prompt: {ex['prompt'][:100]}...")
    print(f"Generation: {ex['generation']}")
    print(f"Ground Truth: {ex['ground_truth']}")
    print("---")

print("\nExamples where format OK but answer wrong:")
for ex in examples["format_ok_answer_wrong"]:
    print(f"Prompt: {ex['prompt'][:100]}...")
    print(f"Generation: {ex['generation']}")
    print(f"Ground Truth: {ex['ground_truth']}")
    print("---")

print("\nExamples where both wrong (format failed):")
for ex in examples["both_wrong"]:
    print(f"Prompt: {ex['prompt'][:100]}...")
    print(f"Generation: {ex['generation']}")
    print(f"Ground Truth: {ex['ground_truth']}")
    print("---")

Category Counts:
{'correct_both': 6, 'format_ok_answer_wrong': 22, 'both_wrong': 1291}

Overall Accuracy (correct both): 0.45%
Format Success Rate: 2.12%

Examples where both correct:
Prompt: 
A conversation between User and Assistant. The User asks a question, and the Assistant solves it. T...
Generation: The bagel cost $4.
  The soup cost 25% more than the bagel, so the soup cost $4 x 1.25 = $5.
  The cake cost half of the price of the bagel, so the cake cost $4 / 2 = $2.
  The total cost of the dinner is $4 (bagel) + $5 (soup) + $2 (cake) = $11.
</think> <answer> $11 </answer>
Ground Truth: 11
---
Prompt: 
A conversation between User and Assistant. The User asks a question, and the Assistant solves it. T...
Generation: The starting value is 20. Let's call the starting value \( S \). So, \( S = 20 \).

The calculation is:
\[ S + \frac{S}{2} \div 5 \]
First, calculate half of the starting value:
\[ \frac{S}{2} = \frac{20}{2} = 10 \]
Now, add this to the starting value:
\[ S + 10 = 20 

# Supervised fine-tuning

## Loading the reasoning traces

In [13]:
# The first thing we need to do is load the reasoning traces we will use for the fine-tuning.

In [14]:
# Load the high-quality math SFT dataset (220k verified traces from DeepSeek-R1)
sft_dataset = load_dataset("open-r1/OpenR1-Math-220k", split="train")

print(f"Loaded {len(sft_dataset)} high-quality math SFT examples")
print("Features:", sft_dataset.features)

# Inspect first example
print("\nFirst example:")
print(json.dumps(sft_dataset[0], indent=2)[:1000] + "..." if len(str(sft_dataset[0])) > 1000 else json.dumps(sft_dataset[0], indent=2))

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Loaded 93733 high-quality math SFT examples
Features: {'problem': Value('string'), 'solution': Value('string'), 'answer': Value('string'), 'problem_type': Value('string'), 'question_type': Value('string'), 'source': Value('string'), 'uuid': Value('string'), 'is_reasoning_complete': List(Value('bool')), 'generations': List(Value('string')), 'correctness_math_verify': List(Value('bool')), 'correctness_llama': List(Value('bool')), 'finish_reasons': List(Value('string')), 'correctness_count': Value('int64'), 'messages': List({'content': Value('string'), 'role': Value('string')})}

First example:
{
  "problem": "## Task B-1.3.\n\nA ship traveling along a river has covered $24 \\mathrm{~km}$ upstream and $28 \\mathrm{~km}$ downstream. For this journey, it took half an hour less than for traveling $30 \\mathrm{~km}$ upstream and $21 \\mathrm{~km}$ downstream, or half an hour more than for traveling $15 \\mathrm{~km}$ upstream and $42 \\mathrm{~km}$ downstream, assuming that both the ship and 

In [15]:
# Let's inspect a few examples by hand to understand the data.

In [16]:
print(type(sft_dataset))
print(len(sft_dataset))

<class 'datasets.arrow_dataset.Dataset'>
93733


In [17]:
index0 = 120

print(type(sft_dataset[index0]))
print(sft_dataset[index0].keys())

<class 'dict'>
dict_keys(['problem', 'solution', 'answer', 'problem_type', 'question_type', 'source', 'uuid', 'is_reasoning_complete', 'generations', 'correctness_math_verify', 'correctness_llama', 'finish_reasons', 'correctness_count', 'messages'])


In [18]:
r1_ex1 = sft_dataset[index0]

In [19]:
print(r1_ex1['problem'])

149. Two equally matched opponents are playing chess. Find the most probable number of wins for any chess player if $2 N$ decisive (without draws) games will be played.


In [20]:
print(r1_ex1['solution'])

Solution. It is known that if the product of the number of trials $\boldsymbol{n}$ and the probability $p$ of the event occurring in one trial is an integer, then the most probable number is

$$
k_{0}=n p .
$$

In the problem at hand, the number of trials $n$ is equal to the number of games played $2 N$; the probability of the event occurring is equal to the probability of winning in one game, i.e., $p=1 / 2$ (by the condition that the opponents are of equal strength).

Since the product $n p=2 N \cdot 1 / 2=N$ is an integer, the sought most probable number $k_{0}$ of games won is $N$.


In [21]:
print(r1_ex1["answer"])

N


In [22]:
# These examples look good but numerical answers, will this be an issue?

## Loading the model

In [23]:
# This follows the code of section 4.1, do we really need to load the model again?

In [24]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-Math-1.5B"
#OUTPUT_DIR = Path("sft_checkpoints")
#OUTPUT_DIR.mkdir(exist_ok=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
#tokenizer.pad_token = tokenizer.eos_token

`torch_dtype` is deprecated! Use `dtype` instead!


In [25]:
# this is to load from a local dir

# model = AutoModelForCausalLM.from_pretrained(
#     "/data/a5-alignment/models/Qwen2.5-Math-1.5B",
#     torch_dtype=torch.bfloat16,
#     attn_implementation="flash_attention_2",
#     )
# tokenizer = AutoTokenizer.from_pretrained("/data/a5-alignment/models/Qwen2.5-Math-1.5B")

## SFT helper methods

In [79]:
# We will implement the functions of section 4.2

In [ ]:
list_of_probs1 = []

for ind1 in range(100):
    list_of_probs1.append(sft_dataset[ind1]['problem'].split("##")[-1].strip()) 

list_of_ans1 = []

for ind1 in range(100):
    list_of_ans1.append(sft_dataset[ind1]['answer'])

list_of_prompts1 = []

for pb in list_of_probs1:
    list_of_prompts1.append(R1_ZERO_PROMPT.format(question=pb))
    

In [38]:
list_of_probs1[0]

'Task B-1.3.\n\nA ship traveling along a river has covered $24 \\mathrm{~km}$ upstream and $28 \\mathrm{~km}$ downstream. For this journey, it took half an hour less than for traveling $30 \\mathrm{~km}$ upstream and $21 \\mathrm{~km}$ downstream, or half an hour more than for traveling $15 \\mathrm{~km}$ upstream and $42 \\mathrm{~km}$ downstream, assuming that both the ship and the river move uniformly.\n\nDetermine the speed of the ship in still water and the speed of the river.'

In [41]:
list_of_prompts1[0]

'\nA conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: Task B-1.3.\n\nA ship traveling along a river has covered $24 \\mathrm{~km}$ upstream and $28 \\mathrm{~km}$ downstream. For this journey, it took half an hour less than for traveling $30 \\mathrm{~km}$ upstream and $21 \\mathrm{~km}$ downstream, or half an hour more than for traveling $15 \\mathrm{~km}$ upstream and $42 \\mathrm{~km}$ downstream, assuming that both the ship and the river move uniformly.\n\nDetermine the speed of the ship in still water and the speed of the river.\nAssistant: <think>\n'

In [36]:
list_of_ans1[0]

'v_{R}=4\\mathrm{~}/\\mathrm{},v_{B}=10\\mathrm{~}/\\mathrm{}'

In [42]:
tokenizer(list_of_ans1[0])

{'input_ids': [85, 15159, 49, 51185, 19, 59, 91550, 90, 93, 4472, 59, 91550, 22655, 85, 15159, 33, 51185, 16, 15, 59, 91550, 90, 93, 4472, 59, 91550, 6257], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
list_token_prompts1 = [tokenizer(x)['input_ids'] for x in list_of_prompts1]
list_token_ans1 = [tokenizer(x)['input_ids'] for x in list_of_ans1]


In [48]:
list_token_prompt_ans1 = []

for i in range(len(list_token_prompts1)):
    list_token_prompt_ans1.append(list_token_prompts1[i] + list_token_ans1[i])

In [51]:
list_lens1 = [len(x) for x in list_token_prompt_ans1]
print(max(list_lens1))

460
